In [35]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
df_activity= pd.read_csv('Customer Flight Activity.csv')
df_history = pd.read_csv('Customer Loyalty History.csv')

In [ ]:
df_activity.info()
df_history.info()

In [ ]:
df_activity.describe()
df_history.describe()


In [31]:
#EDA
#Número de filas y columnas y nombres
print(f"Filas x Columnas (shape): {df_activity.shape}")
print(f"Filas x Columnas (shape): {df_history.shape}")
print(df_activity.columns.tolist())
print(df_history.columns.tolist())

Filas x Columnas (shape): (405624, 10)
Filas x Columnas (shape): (16737, 16)
['Loyalty Number', 'Year', 'Month', 'Flights Booked', 'Flights with Companions', 'Total Flights', 'Distance', 'Points Accumulated', 'Points Redeemed', 'Dollar Cost Points Redeemed']
['Loyalty Number', 'Country', 'Province', 'City', 'Postal Code', 'Gender', 'Education', 'Salary', 'Marital Status', 'Loyalty Card', 'CLV', 'Enrollment Type', 'Enrollment Year', 'Enrollment Month', 'Cancellation Year', 'Cancellation Month']


In [10]:
df_airline = pd.merge(df_activity, df_history, on='Loyalty Number', how='left')

In [ ]:
print("\n========== HISTOGRAMAS NUMÉRICOS ==========")

df_airline.hist(bins=20, figsize=(25,25))
plt.show()

In [ ]:
#LIMPIEZA DE DATOS

In [12]:
#NULOS
tabla_nulos = (df_airline.isnull().sum().reset_index().rename(columns={"index": "columna", 0: "nulos",}))

tabla_nulos["% nulos"] = (tabla_nulos["nulos"] / len(df_airline) * 100).round(2)

tabla_nulos["dtype"] = tabla_nulos["columna"].map(df_airline.dtypes)

In [ ]:
tabla_nulos

In [ ]:
#Los clientes que no volaron tienen NaN en 'Flights Booked', los pasamos a 0
df_airline['Flights Booked'] = df_airline['Flights Booked'].fillna(0)

In [34]:
#Puntos acumulados
df_airline['Points Accumulated'] = pd.to_numeric(df_airline['Points Accumulated'], errors='coerce')

In [ ]:
df_airline.head()

In [14]:
#Año y mes de cancelación
df_airline['Cancellation Year'] = df_airline['Cancellation Year'].astype(object)
df_airline['Cancellation Month'] = df_airline['Cancellation Month'].astype(object)

df_airline['Cancellation Year'] = df_airline['Cancellation Year'].fillna('Activo')
df_airline['Cancellation Month'] = df_airline['Cancellation Month'].fillna('Activo')

df_airline['Cancellation Year'] = pd.to_numeric(df_airline['Cancellation Year'], errors='coerce').astype('Int64').astype(object).fillna('Activo')
df_airline['Cancellation Month'] = pd.to_numeric(df_airline['Cancellation Month'], errors='coerce').astype('Int64').astype(object).fillna('Activo')

In [15]:
#Salario
# 1. Asegurar que es numérico y pasar a positivo
df_airline['Salary'] = pd.to_numeric(df_airline['Salary'], errors='coerce').abs()

# 2. Imputar nulos por la mediana de su nivel educativo
df_airline['Salary'] = df_airline.groupby('Education')['Salary'].transform(lambda x: x.fillna(x.median()))

# 3. Si aún quedara algún nulo residual, usar la mediana global
mediana_general = df_airline['Salary'].median()
df_airline['Salary'] = df_airline['Salary'].fillna(mediana_general).astype('int64')

In [ ]:
#Duplicados
print("\n========== DUPLICADOS ==========")
if df_airline.duplicated().sum() > 0:
    print(f"Se encontraron {df_airline.duplicated().sum()} filas duplicadas.")
    df_airline.drop_duplicates(inplace=True)
else:
    print("¡No hay filas duplicadas!")

df_airline.to_csv('Customer_Airline_Clean.csv', index=False)

In [ ]:
#FASE 2 ANÁLISIS ESTADISTICO
#Variables numéricas claves
cols_num = ['Flights Booked', 'Distance', 'Points Accumulated', 'Salary', 'CLV']
df_airline[cols_num].describe()
df_airline[cols_num].mode().iloc[0]

#Variables categóricas
cols_cat = ['Loyalty Card', 'Education', 'Marital Status', 'Gender']
for col in cols_cat:
    print(f"\n--- Distribución para {col} ---")

    print('Conteo absoluto')
    print (df_airline[col].value_counts())
    print('Porcentaje')
    print(df_airline[col].value_counts(normalize=True) * 100)


In [29]:
df_airline.head(30)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed,...,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,100018,2017,1,3,0,3,1521,152.0,0,0,...,Bachelor,92552,Married,Aurora,7919.20,Standard,2016,8,Activo,Activo
1,100102,2017,1,10,4,14,2030,203.0,0,0,...,College,73479,Single,Nova,2887.74,Standard,2013,3,Activo,Activo
2,100140,2017,1,6,0,6,1200,120.0,0,0,...,College,73479,Divorced,Nova,2838.07,Standard,2016,7,Activo,Activo
3,100214,2017,1,0,0,0,0,0.0,0,0,...,Bachelor,63253,Married,Star,4170.57,Standard,2015,8,Activo,Activo
4,100272,2017,1,0,0,0,0,0.0,0,0,...,Bachelor,91163,Divorced,Star,6622.05,Standard,2014,1,Activo,Activo
5,100301,2017,1,0,0,0,0,0.0,0,0,...,Bachelor,70323,Divorced,Nova,48356.96,Standard,2013,9,Activo,Activo
6,100364,2017,1,0,0,0,0,0.0,0,0,...,Bachelor,76849,Married,Nova,5143.88,Standard,2015,5,Activo,Activo
7,100380,2017,1,0,0,0,0,0.0,0,0,...,Bachelor,69695,Single,Star,2465.62,Standard,2012,10,Activo,Activo
8,100428,2017,1,6,0,6,606,60.0,0,0,...,Bachelor,63478,Married,Aurora,5845.43,Standard,2012,8,Activo,Activo
9,100504,2017,1,0,0,0,0,0.0,0,0,...,Bachelor,75638,Divorced,Nova,8807.61,Standard,2017,7,2018,3


In [18]:
#FASE 3 VISUALIZACIÓN
#Gráficos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 12

In [ ]:
# 1-¿Cómo se distribuye la cantidad de vuelos reservados por mes durante el año?
# Agrupamos los vuelos por mes y sumamos
vuelos_por_mes = df_airline.groupby('Month')['Flights Booked'].sum().reset_index()
sns.barplot(x='Month', y='Flights Booked', data=vuelos_por_mes, hue='Month', palette='Blues_d', legend=False)
plt.title('Total de Vuelos Reservados por Mes durante el Año')
plt.xlabel('Mes del Año')
plt.ylabel('Cantidad de Vuelos Reservados')
plt.xticks(ticks=range(0, 12), labels=['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
plt.show()

In [ ]:
#2-¿Existe una relación entre la distancia de los vuelos y los puntos acumulados por los cliente?
sns.scatterplot(x='Distance', y='Points Accumulated', data=df_airline, alpha=0.5, color='teal')
plt.title('Relación entre Distancia Recorrida y Puntos Acumulados')
plt.xlabel('Distancia del Vuelo (Km/Millas)')
plt.ylabel('Puntos Acumulados')
plt.show()


In [ ]:
#3- ¿Cuál es la distribución de los clientes por provincia o estado?
#Agrupamos por provincia y número de clientes únicos
clientes_provincia = df_airline.groupby('Province')['Loyalty Number'].nunique().sort_values(ascending=False).reset_index()

sns.barplot(x='Loyalty Number', y='Province', data=clientes_provincia, hue='Province', palette='viridis')
plt.title('Distribución Geográfica de Clientes por Provincia')
plt.xlabel('Número de Clientes Únicos')
plt.ylabel('Provincia')
plt.show()



In [ ]:
#Estado
clientes_estado = df_airline.groupby('Country')['Loyalty Number'].nunique().sort_values(ascending=False).reset_index()

sns.barplot(x='Loyalty Number', y='Country', data=clientes_estado, hue='Country', palette='viridis')
plt.title('Distribución Geográfica de Clientes por Estado')
plt.xlabel('Número de Clientes Únicos')
plt.ylabel('Estado')
plt.show()

In [ ]:
#4- ¿Cómo se compara el salario promedio entre los diferentes niveles educativos de los clientes?
# Agrupamos por el salario promedio
educacion = df_airline.groupby('Education')['Salary'].mean().sort_values(ascending=False).index

sns.barplot(x='Salary', y='Education', data=df_airline, hue='Education', order=educacion, palette='magma', errorbar=None)
plt.title('Salario Promedio según el Nivel Educativo del Cliente')
plt.xlabel('Salario Promedio Anual')
plt.ylabel('Nivel Educativo')
plt.show()


In [ ]:

#5-  ¿Cuál es la proporción de clientes con diferentes tipos de tarjetas de fidelidad?
# Agrupamos clientes únicos por tipo de tarjeta
tarjetas = df_airline.groupby('Loyalty Card')['Loyalty Number'].nunique()

plt.pie(tarjetas, labels=tarjetas.index, autopct='%1.1f%%', startangle=90, 
        colors=sns.color_palette('pastel')[0:3], wedgeprops=dict(width=0.4, edgecolor='w'))
plt.title('Proporción de Clientes por Tipo de Tarjeta de Fidelidad')
plt.show()



In [ ]:
#6- ¿Cómo se distribuyen los clientes según su estado civil y género?
#Agrupamos por estado civil y género
df_civil_genero = df_airline.groupby(['Marital Status', 'Gender'])['Loyalty Number'].nunique().reset_index()

sns.barplot(x='Marital Status', y='Loyalty Number', hue='Gender', data=df_civil_genero, palette='muted')
plt.title('Distribución de Clientes por Estado Civil y Género')
plt.xlabel('Estado Civil')
plt.ylabel('Cantidad de Clientes')
plt.legend(title='Género')
plt.show()

In [ ]:
#Fase 4: Evaluación de Diferencias en Reservas de Vuelos por Nivel Educativo
df_diferencias = df_airline[['Flights Booked', 'Education']]
df_diferencias.head()

# Agrupamos por 'Education' y calculamos
analisis_educacion = df_diferencias.groupby('Education')['Flights Booked'].agg(Total_Registros='count',Promedio_Vuelos='mean',Mediana_Vuelos='median',Desviacion_Estandar='std').reset_index()

#Ordenamos
analisis_educacion = analisis_educacion.sort_values(by='Promedio_Vuelos', ascending=False)
print(analisis_educacion)

In [ ]:
#Gráfica
sns.barplot(x='Flights Booked', y='Education', data=df_diferencias, hue ='Education', order=analisis_educacion['Education'], errorbar='ci', palette='muted')
plt.title('Promedio de Vuelos Reservados con Intervalo de Confianza por Nivel Educativo')
plt.xlabel('Promedio de Vuelos Reservados')
plt.ylabel('Nivel Educativo')
plt.show()